# Caracterización de la tirada en FATE2d6

Este notebook analiza matemáticamente la tirada base del sistema:

**2d6 + Atributo + Habilidad**

El objetivo es caracterizar cómo se comporta la tirada en términos de
probabilidad y dificultad, tomando como punto de partida una configuración
concreta del sistema definida en un archivo JSON.

## Objetivos del análisis

- Analizar la distribución de frecuencias del modificador `Atributo + Habilidad`.
- Calcular el valor esperado del modificador a la tirada (E[m]).
- Estudiar la distribución de probabilidad de distintas tiradas posibles.
- Comparar el comportamiento de la tirada frente a la escala de dificultades del sistema.
- Dejar una base reutilizable para analizar otras configuraciones del juego.

## Setup

En esta sección se prepara el entorno de trabajo del notebook:

- se resuelven las rutas del proyecto
- se importan las dependencias necesarias
- se carga la configuración del sistema desde un archivo JSON

Para reutilizar este notebook con otra variante del sistema, basta con cambiar
la ruta del archivo de configuración en la celda siguiente.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline

ROOT = Path().resolve().parent
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from fate2d6.config import load_config
from fate2d6.probability import *

CONFIG_PATH = ROOT / "config" / "base.json"
config = load_config(CONFIG_PATH)

print("LOAD OK")
print(f"Sistema cargado: {config.system_name}")
print(f"Configuración usada: {CONFIG_PATH}")

## Análisis de la Configuración

La configuración cargada define el conjunto de atributos, habilidades y valores
asociados que se usarán para calcular el espacio de modificadores posibles del
sistema.

Al cambiar el archivo JSON de configuración, este notebook puede reutilizarse
para analizar otras variantes de FATE2d6.

In [ ]:
print("Configuración:", config.system_name)
print("Atributos:", config.attribute_names)
print("Distribución de atributos:", config.attribute_value_distribution)
print("Habilidades:", config.skill_names)
print("Distribución de habilidades:", config.skill_value_distribution)

## Distribución del modificador `Atributo + Habilidad`

A continuación se calculan todas las combinaciones posibles entre los valores
de atributos y los valores de habilidades definidos en la configuración
cargada.

El objetivo de esta parte es obtener:

- la frecuencia de cada modificador posible
- la probabilidad asociada a cada uno
- el valor esperado del modificador

Esta distribución describe la estructura interna del sistema para la
configuración elegida. No representa todavía el uso real en mesa, ya que la
frecuencia efectiva de cada combinación dependerá también de la ficción y de
las decisiones de los jugadores.

In [ ]:
distribution = modifier_distribution(config)
probabilities = modifier_probabilities(config)
total = total_modifier_combinations(config)
expected = expected_modifier(config)
mode = modifier_mode(config)
std = modifier_std(config)

rows = []
for modifier, frequency in distribution.items():
    rows.append(
        {
            "Modificador": modifier,
            "Frecuencia": frequency,
            "Probabilidad": probabilities[modifier],
            "Porcentaje": probabilities[modifier] * 100,
        }
    )

df_modifiers = pd.DataFrame(rows)

df_modifiers_display = df_modifiers.copy()
df_modifiers_display["Probabilidad"] = df_modifiers_display["Probabilidad"].map(
    lambda x: f"{x:.3f}"
)
df_modifiers_display["Porcentaje"] = df_modifiers_display["Porcentaje"].map(
    lambda x: f"{x:.1f}%"
)

print("Configuración:", config.system_name)
print("Total de combinaciones:", total)
print("Valor esperado del modificador:", round(expected, 4))
print("Moda:", mode)
print("Desviación típica:", round(std, 4))

x = np.array([-2, -1, 0, 1, 2, 3, 4, 5])
y = np.array([0, 4, 12, 22, 26, 20, 6, 0])

x_smooth = np.linspace(-2, 5, 400)
spline = make_interp_spline(x, y, k=3)
y_smooth = spline(x_smooth)

plt.figure(figsize=(9, 5))

plt.bar(
    df_modifiers["Modificador"].values,
    df_modifiers["Frecuencia"].values,
    alpha=0.6,
    color="darkorange",
    label="Frecuencia",
)

plt.plot(
    x_smooth,
    y_smooth,
    color="steelblue",
    linewidth=2.5,
    label="Tendencia suavizada",
)

plt.axvline(
    expected,
    color="crimson",
    linestyle="--",
    linewidth=2,
    label=f"E[m] = {expected:.2f}",
)

plt.title("Distribución de frecuencias de Atributo + Habilidad")
plt.xlabel("Modificador")
plt.ylabel("Frecuencia")
plt.xticks(np.arange(-2, 6, 1))
plt.xlim(-2, 5)
plt.legend()
plt.grid(alpha=0.2)

plt.show()

df_modifiers_display

### Interpretación de la distribución de modificadores

La tabla y la gráfica anteriores muestran cómo se distribuyen los valores de
`Atributo + Habilidad` en función de la configuración cargada.

Esta distribución refleja:

- qué modificadores son más frecuentes dentro del sistema
- qué valores extremos aparecen con menor probabilidad
- dónde se sitúa el centro de gravedad del sistema (valor esperado)

En general, esta forma describe la estructura matemática del sistema para la
configuración seleccionada. No representa directamente el uso real en partida,
ya que:

- no todas las combinaciones atributo-habilidad son igualmente naturales
- la ficción restringe ciertas acciones
- los jugadores tenderán a optimizar sus elecciones

Por tanto, esta distribución debe interpretarse como una **base estructural**
sobre la que luego se superpone el comportamiento real en mesa.

---

### Caso particular: configuración base

Si se utiliza la configuración base del sistema, definida en `base.json`, los
valores observados son:

- `-1`: 4 casos  
- `0`: 12 casos  
- `1`: 22 casos  
- `2`: 26 casos  
- `3`: 20 casos  
- `4`: 6 casos  

Esto muestra una clara concentración en torno a los valores positivos,
especialmente `+2`, que es el modificador más frecuente.

El valor esperado del modificador es aproximadamente **1.71**, lo que indica que
el sistema, en términos estructurales, está desplazado hacia valores positivos.

Si se modifica la configuración JSON, esta distribución cambiará y deberá
interpretarse de nuevo siguiendo este mismo esquema.

## Baseline estructural: `2d6 + E[m]`

Dado que la distribución de `2d6` es bien conocida en juegos de rol, el interés
del análisis no está en la tirada base aislada, sino en cómo se comporta al
combinarse con los modificadores del sistema.

Como primera aproximación se toma el valor esperado del modificador
`Atributo + Habilidad`, denotado por `E[m]`, y se desplaza la tirada `2d6` por
ese valor.

Este baseline no pretende representar exactamente el uso real en mesa, pero sí
ofrece una referencia estructural útil y conservadora para estudiar la relación
entre la tirada y la escala de dificultades del sistema.

In [ ]:
baseline_distribution = shifted_two_d6_distribution(expected)

rows_baseline = []
for result, probability in sorted(baseline_distribution.items()):
    rows_baseline.append(
        {
            "Resultado": result,
            "Probabilidad": probability,
            "Porcentaje": probability * 100,
        }
    )

df_baseline = pd.DataFrame(rows_baseline)

df_baseline_display = df_baseline.copy()
df_baseline_display["Resultado"] = df_baseline_display["Resultado"].map(lambda x: f"{x:.2f}")
df_baseline_display["Probabilidad"] = df_baseline_display["Probabilidad"].map(lambda x: f"{x:.3f}")
df_baseline_display["Porcentaje"] = df_baseline_display["Porcentaje"].map(lambda x: f"{x:.1f}%")

print("Baseline estructural: 2d6 + E[m]")
print("E[m]:", round(expected, 4))
print("Valor esperado de la tirada baseline:", round(7 + expected, 4))

x_baseline = df_baseline["Resultado"].values
y_baseline = df_baseline["Probabilidad"].values

x_plot = np.concatenate(([x_baseline.min() - 1], x_baseline, [x_baseline.max() + 1]))
y_plot = np.concatenate(([0], y_baseline, [0]))

x_smooth = np.linspace(x_plot.min(), x_plot.max(), 500)
spline = make_interp_spline(x_plot, y_plot, k=3)
y_smooth = spline(x_smooth)

baseline_expected = 7 + expected

plt.figure(figsize=(9, 5))

plt.bar(
    x_baseline,
    y_baseline,
    width=0.35,
    alpha=0.6,
    color="darkorange",
    label="Probabilidad",
)

# plt.plot(
#     x_smooth,
#     y_smooth,
#     color="steelblue",
#     linewidth=2.5,
#     label="Tendencia suavizada",
# )

plt.axvline(
    baseline_expected,
    color="crimson",
    linestyle="--",
    linewidth=2,
    label=f"E[2d6 + E[m]] = {baseline_expected:.2f}",
)

plt.title("Distribución baseline: 2d6 + E[m]")
plt.xlabel("Resultado")
plt.ylabel("Probabilidad")
plt.legend()
plt.grid(alpha=0.2)

plt.show()

df_baseline_display

### Interpretación del baseline `2d6 + E[m]`

Una forma útil de aproximar el comportamiento global del sistema es combinar la
distribución base de `2d6` con un único modificador representativo. En este
caso, se utiliza el valor esperado del modificador `Atributo + Habilidad`,
denotado como `E[m]`.

Este enfoque permite construir un **baseline estructural** de la tirada, que
sirve como referencia para entender dónde se sitúa el centro de la distribución
y cómo se relaciona con la escala de dificultades del sistema. No pretende
describir con exactitud el comportamiento en partida, sino ofrecer una
aproximación conservadora y homogénea sobre la que comparar otros escenarios.

---

### Caso particular: baseline con `E[m]`

Para la configuración base, el valor esperado del modificador es aproximadamente
`1.71`. Al desplazar la distribución de `2d6` por este valor, el centro de la
tirada pasa de `7` a aproximadamente `8.71`.

Esto sitúa el baseline estructural del sistema muy cerca de la dificultad `9`,
que en la escala del juego corresponde al nivel **Bueno / Media**.

Esta proximidad sugiere que, incluso bajo una lectura conservadora, la tirada
tiende a concentrarse en torno a los niveles medios de dificultad del sistema.

Como en el caso anterior, esta interpretación depende de la configuración
utilizada. Si se modifica el JSON de entrada, tanto el valor esperado como la
posición del baseline cambiarán, y deberán analizarse de nuevo en su contexto.

## Distribuciones por modificador fijo

Además del baseline estructural `2d6 + E[m]`, resulta útil estudiar cómo se
comporta la tirada cuando se utiliza un modificador fijo concreto.

Este análisis permite observar de forma más precisa el efecto de cada nivel de
competencia sobre la distribución final de resultados. En particular, permite
comparar entre sí las tiradas:

- `2d6 - 1`
- `2d6 + 0`
- `2d6 + 1`
- `2d6 + 2`
- `2d6 + 3`
- `2d6 + 4`

Estas distribuciones son especialmente útiles para interpretar qué significa en
mesa tirar con un modificador concreto y cómo se desplaza la tirada respecto de
la escala de dificultades del sistema.

In [ ]:
fixed_modifiers = [-1, 0, 1, 2, 3, 4]

summary_rows = []

plt.figure(figsize=(10, 6))

for modifier in fixed_modifiers:
    shifted_distribution = shifted_two_d6_distribution(modifier)

    x = np.array([1 + modifier, *shifted_distribution.keys(), 13 + modifier])
    y = np.array([0, *shifted_distribution.values(), 0])

    x_smooth = np.linspace(x.min(), x.max(), 500)
    spline = make_interp_spline(x, y, k=3)
    y_smooth = spline(x_smooth)

    plt.plot(
        x_smooth,
        y_smooth,
        linewidth=2,
        label=f"2d6 {modifier:+d}",
    )

    summary_rows.append(
        {
            "Modificador": f"{modifier:+d}",
            "Valor esperado": round(expected_shifted_two_d6(modifier), 3),
            "Moda": 7 + modifier,
        }
    )

plt.title("Distribuciones de probabilidad para 2d6 + modificador fijo")
plt.xlabel("Resultado")
plt.ylabel("Probabilidad")
plt.xticks(np.arange(0, 18, 1))
plt.xlim(0, 17)
plt.legend()
plt.grid(alpha=0.2)

plt.show()

df_fixed_modifiers = pd.DataFrame(summary_rows)
df_fixed_modifiers

## Interpretación de las distribuciones por modificador fijo

Las curvas anteriores muestran cómo se desplaza la distribución de `2d6` cuando
se añade un modificador fijo.

De forma general, cada incremento de `+1`:

- desplaza la distribución una unidad hacia la derecha
- aumenta el valor esperado de la tirada en una unidad
- hace más accesibles las dificultades altas
- reduce la probabilidad relativa de quedarse en resultados bajos

Esto permite interpretar de forma muy directa el impacto práctico de cada
modificador sobre la tirada.

---

### Lectura del caso concreto

En este sistema, los modificadores posibles van de `-1` a `+4`. Esto significa
que el centro de la tirada puede desplazarse desde `6` hasta `11`:

- con `-1`, la tirada orbita en torno a `6`
- con `+0`, orbita en torno a `7`
- con `+1`, orbita en torno a `8`
- con `+2`, orbita en torno a `9`
- con `+3`, orbita en torno a `10`
- con `+4`, orbita en torno a `11`

Esto ya permite anticipar una conclusión importante: los modificadores `+2` y
superiores sitúan el centro de la tirada directamente sobre la zona media-alta
de la escala de dificultades del sistema.

## Probabilidad de alcanzar o superar una dificultad

Una vez obtenidas las distribuciones por modificador fijo, el siguiente paso es
calcular la probabilidad de alcanzar o superar cada nivel de dificultad del
sistema.

Este análisis permite traducir la forma abstracta de las distribuciones a una
lectura directamente útil en mesa: qué posibilidades reales tiene un personaje,
con un modificador dado, de superar una tirada objetivo concreta.

In [ ]:
difficulty_thresholds = list(range(5, 17))
modifier_labels = [-1, 0, 1, 2, 3, 4]

difficulty_rows = []

for modifier in modifier_labels:
    distribution = shifted_two_d6_distribution(modifier)

    row = {"Modificador": f"{modifier:+d}"}
    for difficulty in difficulty_thresholds:
        row[f"{difficulty}+"] = probability_at_least(distribution, difficulty)
    difficulty_rows.append(row)

baseline_distribution = shifted_two_d6_distribution(expected)
baseline_row = {"Modificador": "E[m]"}
for difficulty in difficulty_thresholds:
    baseline_row[f"{difficulty}+"] = probability_at_least(
        baseline_distribution,
        difficulty,
    )
difficulty_rows.append(baseline_row)

df_difficulties = pd.DataFrame(difficulty_rows)

df_difficulties_display = df_difficulties.copy()
for column in df_difficulties_display.columns[1:]:
    df_difficulties_display[column] = df_difficulties_display[column].map(
        lambda x: f"{x * 100:.1f}%"
    )

heatmap_df = df_difficulties.set_index("Modificador").copy()

for column in heatmap_df.columns:
    heatmap_df[column] = heatmap_df[column].astype(float)

plt.figure(figsize=(12, 4))
plt.imshow(heatmap_df.values, aspect="auto")

plt.colorbar(label="Probabilidad de superar la dificultad")
plt.xticks(
    ticks=np.arange(len(heatmap_df.columns)),
    labels=heatmap_df.columns,
)
plt.yticks(
    ticks=np.arange(len(heatmap_df.index)),
    labels=heatmap_df.index,
)

for i in range(heatmap_df.shape[0]):
    for j in range(heatmap_df.shape[1]):
        value = heatmap_df.iloc[i, j]
        plt.text(
            j,
            i,
            f"{value * 100:.0f}%",
            ha="center",
            va="center",
            fontsize=8,
        )

plt.title("Probabilidad de alcanzar o superar cada dificultad")
plt.xlabel("Dificultad objetivo")
plt.ylabel("Modificador")

plt.show()

df_difficulties_display

### Interpretación de la tabla de dificultades

La tabla anterior muestra, para cada modificador, la probabilidad de alcanzar o
superar cada dificultad objetivo.

Esta representación permite leer el sistema de forma directamente operativa:

- qué dificultades son rutinarias para cada modificador
- a partir de qué punto una dificultad pasa a ser incierta
- en qué zona de la escala empieza a sentirse la presión real de la tirada

La fila `E[m]` actúa aquí como baseline estructural y permite comparar el
comportamiento agregado del sistema con los casos concretos de modificador fijo.

In [ ]:
key_difficulties = [8, 9, 10, 12, 14]
modifier_order = [-1, 0, 1, 2, 3, 4]

plt.figure(figsize=(9, 5))

for difficulty in key_difficulties:
    y_values = []
    for modifier in modifier_order:
        distribution = shifted_two_d6_distribution(modifier)
        y_values.append(probability_at_least(distribution, difficulty))

    plt.plot(
        modifier_order,
        y_values,
        marker="o",
        linewidth=2,
        label=f"{difficulty}+",
    )

baseline_values = []
for difficulty in key_difficulties:
    baseline_values.append(probability_at_least(baseline_distribution, difficulty))

plt.axvline(
    expected,
    linestyle="--",
    linewidth=2,
    label=f"E[m] = {expected:.2f}",
)

plt.title("Probabilidad de superar dificultades clave según el modificador")
plt.xlabel("Modificador")
plt.ylabel("Probabilidad")
plt.xticks(modifier_order)
plt.legend()
plt.grid(alpha=0.2)

plt.show()